# 10. Latest commission by calendar date

**Medium**

Return each employee latest commission by calendar date. monthlastdate uses dd-MMM-yyyy format. Keep the original string in the output. Dates are unique per employee.


In [0]:
%run ./_setup $exercise_id="10"


**Your solution:** add imports and return your result from `solve`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def solve(input_df):
    # Write your solution here.
    result_df=(
        input_df
        .withColumn("date_parsed",F.to_date("monthlastdate","dd-MMM-yyyy"))
        .withColumn(
            "rank",
            F.row_number().over(
                Window.partitionBy("empid")
                .orderBy(F.col("date_parsed").desc())
                )
            )
        .filter(F.col("rank")==1)
        .drop("rank","date_parsed")
    )
    return result_df


solve(input_df).show()


In [0]:
from pyspark.sql import functions as F

def solve(input_df):
    # Write your solution here.
    max_date_df=(
        input_df
        .withColumn("date_parsed",F.to_date("monthlastdate","dd-MMM-yyyy"))
        .groupBy("empid")
        .agg(F.max("date_parsed").alias("max_date"))
    )
    input_df= input_df.withColumn("date_parsed",F.to_date("monthlastdate","dd-MMM-yyyy"))
    
    return (
        input_df.alias("a")
        .join(
            max_date_df.alias("b"),
            (F.col("a.date_parsed")== F.col("b.max_date")) &
            (F.col("a.empid")==F.col("b.empid")),
            how="inner"
            )
        .select("a.empid","a.commissionamt","a.monthlastdate")
    )

solve(input_df).show()


**Check:** run below to test all cases and save progress.


In [0]:
check_report = check_solution(exercise_id, solve, spark)


In [0]:
show_progress(exercise_id, spark)
